# order_reviews の重複調査

## 背景

`src/check_relationships.py` の検証で、以下が見つかった。

- `order_reviews(order_id) -> orders(order_id)` は 1:1 を想定していたが、実際は 1:N (最大 3 件)
- `review_id` 単独では一意にならない (重複 814 件)。主キー候補は `(review_id, order_id)`

## 目的

1注文に複数のレビューが付いているケースについて実際のデータ構造を確認し、後続の整形・集約方法を決める材料にする。

In [1]:
from pathlib import Path

import pandas as pd

# notebook はこのファイルのあるディレクトリ (notebooks/) をカレントとして実行される想定
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
reviews.shape

(99224, 7)

## 1. 規模

`order_id` が 2 行以上出現する注文を抽出し、件数を確認する。

In [2]:
multi = (
    reviews[reviews.duplicated("order_id", keep=False)]
    .sort_values(["order_id", "review_creation_date"])
)
print(f"rows: {len(multi):,} / orders: {multi['order_id'].nunique():,}")

# 注文あたりのレビュー件数の分布
multi.groupby("order_id").size().value_counts().rename_axis("reviews_per_order").rename("orders")

rows: 1,098 / orders: 547


reviews_per_order
2    543
3      4
Name: orders, dtype: int64

## 2. 実際の行を見る

先頭のいくつかの注文について、レビュー行を並べて比較する。

In [3]:
sample_orders = multi["order_id"].drop_duplicates().head(6)
multi[multi["order_id"].isin(sample_orders)].drop(columns="review_comment_title")

,review_id,order_id,review_score,review_comment_message,review_creation_date,review_answer_timestamp
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,"Estou acostumada a comprar produtos pelo baratheon. Super confiável, transpa...",2017-08-25 00:00:00,2017-08-29 21:45:57
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PEDIDO NAO RECEBI AINDA E OUTRO ...,2018-03-21 00:00:00,2018-03-22 01:32:08
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,ja reclamei varias vezes e ate hoje não sei onde foi parar o produto que com...,2018-03-29 00:00:00,2018-03-30 03:16:19
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11


## 3. 同じ注文のレビュー同士はどれくらい似ているか

完全なコピーなのか、別々の回答なのかを確認する。

In [4]:
by_order = multi.groupby("order_id")

pd.Series(
    {
        "orders": multi["order_id"].nunique(),
        "same_review_id": (by_order["review_id"].nunique() == 1).sum(),
        "same_score": (by_order["review_score"].nunique() == 1).sum(),
        "same_creation_date": (by_order["review_creation_date"].nunique() == 1).sum(),
        # コメント無し (NaN) 同士も「同じ」とみなす
        "same_message": (by_order["review_comment_message"].nunique(dropna=False) == 1).sum(),
    }
)

orders                547
same_review_id          0
same_score            345
same_creation_date    155
same_message          317
dtype: int64

## 4. その review_id は他の注文にも付いているか

`review_id` の重複 (814 件) との関係を確認する。

In [5]:
shared_review_ids = reviews.loc[reviews.duplicated("review_id", keep=False), "review_id"]
n_orders = multi.loc[multi["review_id"].isin(shared_review_ids), "order_id"].nunique()
print(f"他の注文と review_id を共有しているレビューを持つ注文: {n_orders:,} / {multi['order_id'].nunique():,}")

他の注文と review_id を共有しているレビューを持つ注文: 458 / 547


## 5. review_id が複数の注文に付いているケースの規模

注文・顧客の情報と結合して、`review_id` が 2 行以上出現するレビューを抽出する。

In [6]:
orders = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv",
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"],
)
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")

shared = (
    reviews[reviews.duplicated("review_id", keep=False)]
    .merge(orders, on="order_id")
    .merge(customers, on="customer_id")
    .sort_values(["review_id", "order_purchase_timestamp"])
)
print(
    f"rows: {len(shared):,} / review_ids: {shared['review_id'].nunique():,}"
    f" / orders: {shared['order_id'].nunique():,}"
)

# review_id あたりの注文数の分布
shared.groupby("review_id").size().value_counts().sort_index().rename_axis("orders_per_review").rename("review_ids")

rows: 1,603 / review_ids: 789 / orders: 1,412


orders_per_review
2    764
3     25
Name: review_ids, dtype: int64

## 6. 行の中身は同じか・同じ顧客か

`order_id` 以外のレビュー列がすべて同じなら、1 件の回答が複数の注文に複製されていると考えられる。

In [7]:
by_review = shared.groupby("review_id")
review_columns = [c for c in reviews.columns if c != "order_id"]

pd.Series(
    {
        "review_ids": shared["review_id"].nunique(),
        # NaN 同士も「同じ」とみなす
        "identical_except_order_id": (by_review[review_columns].nunique(dropna=False).max(axis=1) == 1).sum(),
        "same_customer_unique_id": (by_review["customer_unique_id"].nunique() == 1).sum(),
    }
)

review_ids                   789
identical_except_order_id    789
same_customer_unique_id      789
dtype: int64

In [8]:
DISPLAY_COLUMNS = [
    "review_id",
    "order_id",
    "customer_unique_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "review_score",
    "review_creation_date",
]

sample_reviews = shared["review_id"].drop_duplicates().head(3)
shared.loc[shared["review_id"].isin(sample_reviews), DISPLAY_COLUMNS]

,review_id,order_id,customer_unique_id,order_status,order_purchase_timestamp,order_delivered_customer_date,review_score,review_creation_date
752,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,f30856ad31d3e74253a3f4ccef670648,delivered,2018-02-02 18:01:08,2018-02-26 18:18:45,1,2018-03-07 00:00:00
464,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,f30856ad31d3e74253a3f4ccef670648,delivered,2018-02-02 18:01:10,2018-03-11 11:32:24,1,2018-03-07 00:00:00
1017,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,91ffaf5eb7f4bd6a48b07f3546fb7a99,delivered,2017-09-04 15:17:11,2017-09-18 19:43:06,5,2017-09-21 00:00:00
1460,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,91ffaf5eb7f4bd6a48b07f3546fb7a99,delivered,2017-09-04 15:17:12,2017-09-20 22:24:03,5,2017-09-21 00:00:00
930,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,ba16ab3721ffd6b57fc87a7a4f6a4fcf,delivered,2018-02-15 13:09:28,2018-02-24 00:33:11,1,2018-03-07 00:00:00
1498,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,ba16ab3721ffd6b57fc87a7a4f6a4fcf,delivered,2018-02-15 13:09:29,2018-03-06 21:39:06,1,2018-03-07 00:00:00


## 7. 同じレビューが付いた注文同士の関係

購入時刻が近ければ、1 回の購入 (チェックアウト) が複数の注文に分割されたと考えられる。
分割の理由を探るため、注文ごとの販売者も比較する。

In [9]:
purchase_span = by_review["order_purchase_timestamp"].agg(lambda s: s.max() - s.min())

pd.Series(
    {
        "same_second": (purchase_span == pd.Timedelta(0)).sum(),
        "within_1_day": (purchase_span <= pd.Timedelta("1D")).sum(),
        "over_1_day": (purchase_span > pd.Timedelta("1D")).sum(),
        "median": purchase_span.median(),
        "max": purchase_span.max(),
    }
)

same_second                   254
within_1_day                  748
over_1_day                     41
median            0 days 00:00:01
max             127 days 03:24:15
dtype: object

In [10]:
# 注文ごとの販売者の組 (明細の無い注文は含まれない)
sellers_per_order = items.groupby("order_id")["seller_id"].agg(frozenset)


def compare_sellers(order_ids: pd.Series) -> str:
    """同じレビューが付いた注文同士で、販売者の組が同じかを判定する."""
    seller_sets = order_ids.map(sellers_per_order).dropna()
    if len(seller_sets) < len(order_ids):
        return "items_missing"
    return "same_sellers" if seller_sets.nunique() == 1 else "different_sellers"


by_review["order_id"].agg(compare_sellers).value_counts()

order_id
different_sellers    500
same_sellers         237
items_missing         52
Name: count, dtype: int64

## 8. 例外: 購入時刻が 1 日以上離れているケース

In [11]:
far_reviews = purchase_span[purchase_span > pd.Timedelta("1D")].index[:3]
shared.loc[shared["review_id"].isin(far_reviews), DISPLAY_COLUMNS]

,review_id,order_id,customer_unique_id,order_status,order_purchase_timestamp,order_delivered_customer_date,review_score,review_creation_date
851,047fd109ced39e02296f6aeb74f6a6f1,236e6ec6171c1870d4bcf4ccfad87f49,d05c44a138277ad325d915c6b7ccbcdf,canceled,2018-08-21 15:12:31,NaT,4,2018-07-31 00:00:00
210,047fd109ced39e02296f6aeb74f6a6f1,a89abace0dcc01eeb267a9660b5ac126,d05c44a138277ad325d915c6b7ccbcdf,canceled,2018-09-06 18:45:47,NaT,4,2018-07-31 00:00:00
1203,08528f70f579f0c830189efc523d2182,53c71d3953507c6239ff73917ed358c9,46450c74a0d8c5ca9395da1daac6c120,delivered,2018-07-24 20:41:01,2018-08-02 18:41:16,1,2018-08-03 00:00:00
634,08528f70f579f0c830189efc523d2182,03310aa823a66056268a3bab36e827fb,46450c74a0d8c5ca9395da1daac6c120,canceled,2018-08-07 16:33:59,NaT,1,2018-08-03 00:00:00
120,08528f70f579f0c830189efc523d2182,7813842ae95e8c497fc0233232ae815a,46450c74a0d8c5ca9395da1daac6c120,canceled,2018-08-17 20:06:36,NaT,1,2018-08-03 00:00:00
659,1775bd097716d67420d128df008252eb,0b3ca8c9e3b3199306adb467fcd0e982,d526c2df957cc6b177070216427c299d,delivered,2018-07-16 13:34:31,2018-08-03 18:29:09,5,2018-08-04 00:00:00
1076,1775bd097716d67420d128df008252eb,04a2ba381317648726f46210a46ece95,d526c2df957cc6b177070216427c299d,canceled,2018-08-09 09:13:44,NaT,5,2018-08-04 00:00:00


レビュー作成日が購入日より前という、時系列が矛盾した行が見られる。テーブル全体で件数を確認する。

In [12]:
reviews_with_orders = reviews.merge(orders, on="order_id")
# review_creation_date は時刻が 00:00:00 固定のため、購入日 (日付のみ) と比較する
created_before_purchase = (
    pd.to_datetime(reviews_with_orders["review_creation_date"])
    < reviews_with_orders["order_purchase_timestamp"].dt.normalize()
)
is_shared = reviews_with_orders["review_id"].duplicated(keep=False)
print(
    f"レビュー作成日が購入日より前の行: {created_before_purchase.sum()}"
    f" (うち review_id が重複している行: {(created_before_purchase & is_shared).sum()})"
)

reviews_with_orders.loc[created_before_purchase, "order_status"].value_counts()

レビュー作成日が購入日より前の行: 64 (うち review_id が重複している行: 40)


order_status
canceled     57
delivered     6
shipped       1
Name: count, dtype: int64

## 考察

### A. `order_id` の重複 (1 注文に複数レビュー: 547 注文)

- 別々の回答である。`review_id` はすべて異なり、スコアや作成日が異なるものも多い。
- 日を空けた再回答 (例 `02355020…`: 配送トラブルが続きスコアが 3 → 1) や、
  ほぼ同時の二重回答 (例 `0176a684…`: 数十秒差で同じ内容) が見られる。

### B. `review_id` の重複 (1 レビューが複数注文: 789 review_id / 1,412 注文)

- 789 件すべてで、`order_id` 以外の列が完全に一致し、顧客 (`customer_unique_id`) も同じ。
  **1 件の回答が、複数の注文に複製されて紐付いている。**
- 大半 (748 / 789) は購入時刻が 1 日以内で、中央値は 1 秒差。
  購入時刻が非常に近い複数注文に同じレビューが紐付いており、複数注文をまとめた購入単位に対するレビューと考えるのが自然。
  - 販売者が異なる組が 500 件と多いが、同じ販売者の組も 237 件あり、分割の理由は販売者だけでは説明できない。
- 例外として、購入時刻が 1 日以上離れたもの (41 件) がある。
  ここには「レビュー作成日が購入日より前」という時系列が矛盾した行が含まれる。
  - テーブル全体では 64 行あり、うち 57 行がキャンセルされた注文。紐付けの不具合の可能性がある。

### まとめ

`order_reviews` の粒度は「1 注文 1 レビュー」ではなく、**「アンケート回答 (`review_id`) × 対象の注文 (`order_id`)」**。
A と B は別の現象で、両方を考慮して整形する必要がある。

## 整形方針の候補 (整形方針は、後続の分析単位・特徴量設計に応じて決定する)

| 分析の単位 | 問題 | 方針の候補 |
|---|---|---|
| 注文単位 (注文とレビューを結合する) | A により 1 注文が複数行になる | `order_id` ごとに 1 件に絞る。最新の回答 (`review_answer_timestamp` が最大) か、最初の回答を採用する |
| レビュー単位 (スコア分布など) | B により同じ回答を複数回数える | `review_id` で重複を除く |
| 共通 | レビュー作成日が購入日より前の行 (64 行) | 除外する / フラグを立てて残す |

「order_reviews は単純な1注文1レビューのテーブルではないことを確認した」

↓

「review_id × order_id がレコードの粒度であり、同一レビューが複数注文に紐づくケースもある」

↓

「したがって、レビューを使う際は分析単位に応じた重複排除・集約が必要」